In [5]:
import numpy as np
import os
import energycal
from tqdm import tqdm

In [3]:
path = 'data/testcoinc.bin'
row_size = 16
num_rows = os.path.getsize(path) // row_size
dtypes = np.dtype([('time', np.int64), 
                    ('energy', np.float32), 
                    ('id', np.int32)])
data = np.memmap(path, dtype=dtypes, mode='r', shape=(num_rows,))
print(data[0:3])
print(num_rows)
photopeaks = np.fromfile('photopeaks.bin')
escalars = energycal.obtainscalars(photopeaks)


[(6471509218573, 21.53748  , 53) (6471509219847,  6.9428177, 63)
 (6471509219762,  4.1113434, 45)]
54415706


In [16]:
leakagematrix = np.zeros((64, 64)) # Models leakage from crystal at row -> col
leakagecounts = np.zeros((64, 64))
for i in tqdm(range(0, num_rows, 1000000)):
    chunk = np.array(data[i:i+1000000])
    chunk = chunk[chunk['id'] > 127]
    chunk = chunk[chunk['id'] < 192]
    chunk['energy'] = energycal.convertenergy(chunk['energy'], escalars[chunk['id']])
    chunk['id'] = chunk['id'] % 64
    lasttime = chunk[0]['time']
    sidx, eidx = 0, 0
    neventsthisgroup = 0
    for i in range(len(chunk)):
        if chunk[i]['time'] - lasttime > 10000:
            if np.any(np.isclose(chunk[sidx:eidx]['energy'], 511.0, atol = 20.0)):
                centralid = chunk[sidx:eidx]['id'][np.where(np.isclose(chunk[sidx:eidx]['energy'], 511.0, atol = 20.0))[0][0]]
                leakagematrix[centralid, chunk[sidx:eidx]['id']] += chunk[sidx:eidx]['energy']
                leakagecounts[centralid, chunk[sidx:eidx]['id']] += 1
            sidx = eidx
            lasttime = chunk[i]['time']
            neventsthisgroup = 0
        eidx += 1
        neventsthisgroup += 1
leakagematrix = leakagematrix / leakagecounts
for i in range(64): # Normalize by central crystal
    leakagematrix[:, i] = leakagematrix[:, i] / leakagematrix[i, i]
leakagematrix[np.isnan(leakagematrix)] = 0

100%|██████████| 55/55 [00:32<00:00,  1.67it/s]
/tmp/ipykernel_255058/1846802389.py:23: RuntimeWarning: invalid value encountered in divide
  leakagematrix = leakagematrix / leakagecounts


In [18]:
leakagematrix = leakagematrix.astype(np.float32)
leakagematrix.tofile('leakagematrix.bin')